In [ ]:
from agents import Agent, WebSearchTool, trace, Runner, gen_trace_id, function_tool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import asyncio
import requests
import os
from typing import Dict
from IPython.display import display, Markdown

In [ ]:
load_dotenv(override=True)

In [ ]:
INSTRUCTIONS = "You are a research assistant. Given a search term, you search the web for that term and \
produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 \
words. Capture the main points. Write succintly, no need to have complete sentences or good \
grammar. This will be consumed by someone synthesizing a report, so it's vital you capture the \
essence and ignore any fluff. Do not include any additional commentary other than the summary itself."

search_agent=Agent(
    name="Search agent",
    instructions=INSTRUCTIONS,
    tools=[WebSearchTool(search_context_size="low")],
    model="gpt-4o-mini",
    model_settings=ModelSettings(tool_choice="required")
)


In [ ]:
message = "Latest AI Agent frameworks in 2025"

with trace("Search"):
    result=await Runner.run(search_agent,message)

display(Markdown(result.final_output))

In [6]:
### We will now use Structured Outputs, and include a description of the fields
HOW_MANY_SEARCHES = 3

INSTRUCTIONS= f"You are a helpful research assistant. Given a query, come up with a set of web searches \
to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for."

class WebSearchItem(BaseModel):
    reason:str = Field(description="Your reasoning for why this search is important to the query.")
    query:str=Field(description="The search term to use for the web search.")

class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")

planner_agent=Agent(
    name="PlannerAgent",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=WebSearchPlan
)

In [7]:
message="Latest AI Agent frameworks in 2025"

with trace("Search"):
    result = await Runner.run(planner_agent,message)
    print(result.final_output)

searches=[WebSearchItem(reason='To find information on the most recent AI agent frameworks and their features released in 2025.', query='latest AI agent frameworks 2025'), WebSearchItem(reason='To explore technology reviews and expert opinions on the performance and capabilities of new AI agent frameworks in 2025.', query='reviews of AI agent frameworks 2025'), WebSearchItem(reason='To gather news articles or press releases about emerging AI technologies and frameworks introduced in 2025.', query='new AI agent technologies 2025')]


In [11]:
@function_tool
def send_email(subject: str, html_body: str) -> str:
    """Send out an email with the given subject and HTML body """
    
    from_email = "Mr <onboarding@resend.dev>"
    to_email = "moumita.ray.soft8@gmail.com"
    RESEND_API_KEY = os.environ.get("RESEND_API_KEY")
    
    headers = {
        "Authorization": f"Bearer {RESEND_API_KEY}",
        "Content-Type": "application/json"
    }
    
    payload = {
        "from": from_email,
        "to": [to_email],
        "subject": subject,
        "html": html_body
    }
    
    response = requests.post("https://api.resend.com/emails", json=payload, headers=headers)
    
    if response.status_code == 202:
        return "Email sent successfully"
    else:
        return f"Email failed to send due to {response.text}"

In [12]:
send_email

FunctionTool(name='send_email', description='Send out an email with the given subject and HTML body', params_json_schema={'properties': {'subject': {'title': 'Subject', 'type': 'string'}, 'html_body': {'title': 'Html Body', 'type': 'string'}}, 'required': ['subject', 'html_body'], 'title': 'send_email_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x0000018F066FDB20>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None)

In [13]:
INSTRUCTIONS = """You are able to send a nicely formatted HTML email based on a detailed report.
You will be provided with a detailed report. You should use your tool to send one email, providing the 
report converted into clean, well presented HTML with an appropriate subject line."""

email_agent = Agent(
    name="Email agent",
    instructions=INSTRUCTIONS,
    tools = [send_email],
    model="gpt-4o-mini"
)

In [ ]:
INSTRUCTIONS = (
    "You are a senior researcher tasked with writing a cohesive report for a research query. "
    "You will be provided with the original query, and some initial research done by a research assistant.\n"
    "You should first come up with an outline for the report that describes the structure and "
    "flow of the report. Then, generate the report and return that as your final output.\n"
    "The final output should be in markdown format, and it should be lengthy and detailed. Aim "
    "for 5-10 pages of content, at least 1000 words."
)

class ReportData(BaseModel):
    short_summary:str= Field(description="A short 2-3 sentence summary of the findings.")
    markdown_report:str = Field(description="The final report")
    follow_up_questions:list[str] = Field(description="Suggested topics to research further")

writer_agent = Agent(
    name="WriterAgent",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=ReportData
)

TypeError: unsupported operand type(s) for -: 'type' and 'FieldInfo'